Import libraries and configure paths

In [9]:
import numpy as np
import pandas as pd
import joblib

from pathlib import Path
from IPython.display import display

from sklearn.linear_model import Ridge
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import KFold, cross_validate, cross_val_predict

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

project_dir = Path(
    r"C:\Users\U S E R\Desktop\ML Assigment\project"
)

models_dir = project_dir / "models"

results_dir = (
    project_dir / "data" / "processed" / "model_comparisons"
)

models_dir.mkdir(parents=True, exist_ok=True)
results_dir.mkdir(parents=True, exist_ok=True)

print("Setup completed.")

Setup completed.


Load processed training data

In [15]:
train_path = (
    project_dir / "data" / "processed" /
    "math_train_processed.csv"
)

train_df = pd.read_csv(train_path)

X_train = train_df.drop(columns=["G3"])
y_train = train_df["G3"]

assert X_train.notna().all().all()
assert y_train.notna().all()

print("Training samples:", X_train.shape[0])
print("Features:", X_train.shape[1])
train_df.head()

Training samples: 316
Features: 58


,num__age,num__Medu,num__Fedu,num__traveltime,num__studytime,num__failures,num__famrel,num__freetime,num__goout,num__Dalc,...,cat__activities_yes,cat__nursery_no,cat__nursery_yes,cat__higher_no,cat__higher_yes,cat__internet_no,cat__internet_yes,cat__romantic_no,cat__romantic_yes,G3
0,-0.5,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,0.0,0.0,...,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,12
1,-0.5,-0.5,0.0,1.0,-1.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,14
2,-0.5,-1.0,0.0,0.0,0.0,3.0,0.0,0.0,1.0,0.0,...,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0
3,-0.5,0.5,0.0,0.0,1.0,0.0,-1.0,1.0,0.5,1.0,...,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,9
4,-0.5,-0.5,-2.0,1.0,-1.0,0.0,-1.0,0.0,-0.5,0.0,...,1.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,8


Define Ridge Regression experiments

In [16]:
ridge_experiments = {
    "Ridge-1": 0.01,
    "Ridge-2": 0.1,
    "Ridge-3": 1.0,
    "Ridge-4": 10.0,
    "Ridge-5": 100.0
}

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "MAE": "neg_mean_absolute_error",
    "MSE": "neg_mean_squared_error",
    "R2": "r2"
}

display(pd.DataFrame({
    "Experiment": ridge_experiments.keys(),
    "Alpha": ridge_experiments.values()
}))

,Experiment,Alpha
0,Ridge-1,0.01
1,Ridge-2,0.10
2,Ridge-3,1.00
3,Ridge-4,10.00
4,Ridge-5,100.00


Train and evaluate every Ridge configuration

In [17]:
results = []
fold_results = []

actual = y_train.to_numpy()
actual_binary = (actual >= 10).astype(int)

for name, alpha in ridge_experiments.items():

    model = Ridge(alpha=alpha)

    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        return_train_score=True
    )

    predictions = cross_val_predict(
        Ridge(alpha=alpha),
        X_train,
        y_train,
        cv=cv
    )

    rounded_predictions = np.clip(
        np.rint(predictions), 0, 20
    ).astype(int)

    predicted_binary = (predictions >= 10).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        actual_binary,
        predicted_binary,
        labels=[0, 1]
    ).ravel()

    train_rmse = np.sqrt(-scores["train_MSE"]).mean()
    val_rmse = np.sqrt(-scores["test_MSE"]).mean()

    results.append({
        "Experiment": name,
        "Alpha": alpha,

        "Train MAE": -scores["train_MAE"].mean(),
        "Validation MAE": -scores["test_MAE"].mean(),

        "Train MSE": -scores["train_MSE"].mean(),
        "Validation MSE": -scores["test_MSE"].mean(),

        "Train RMSE": train_rmse,
        "Validation RMSE": val_rmse,

        "Train R2": scores["train_R2"].mean(),
        "Validation R2": scores["test_R2"].mean(),

        "RMSE Gap": val_rmse - train_rmse,

        "Exact Grade Accuracy (%)": (
            np.mean(rounded_predictions == actual) * 100
        ),

        "Within 1 Grade (%)": (
            np.mean(np.abs(actual - predictions) <= 1) * 100
        ),

        "Within 2 Grades (%)": (
            np.mean(np.abs(actual - predictions) <= 2) * 100
        ),

        "Pass/Fail Accuracy (%)": (
            accuracy_score(actual_binary, predicted_binary) * 100
        ),

        "Precision": precision_score(
            actual_binary, predicted_binary,
            zero_division=0
        ),

        "Recall": recall_score(
            actual_binary, predicted_binary,
            zero_division=0
        ),

        "Specificity": (
            tn / (tn + fp) if tn + fp > 0 else 0
        ),

        "F1 Score": f1_score(
            actual_binary, predicted_binary,
            zero_division=0
        ),

        "TP": tp,
        "TN": tn,
        "FP": fp,
        "FN": fn
    })

    for fold in range(5):
        fold_results.append({
            "Experiment": name,
            "Fold": fold + 1,
            "Train RMSE": np.sqrt(-scores["train_MSE"][fold]),
            "Validation RMSE": np.sqrt(-scores["test_MSE"][fold]),
            "Train MAE": -scores["train_MAE"][fold],
            "Validation MAE": -scores["test_MAE"][fold],
            "Train R2": scores["train_R2"][fold],
            "Validation R2": scores["test_R2"][fold]
        })

ridge_results_df = pd.DataFrame(results)
ridge_fold_df = pd.DataFrame(fold_results)

ridge_results_df = ridge_results_df.sort_values(
    "Validation RMSE"
).reset_index(drop=True)

print("All Ridge experiments completed.")

All Ridge experiments completed.


Regression error comparison table

In [18]:
regression_columns = [
    "Experiment",
    "Alpha",
    "Train MAE",
    "Validation MAE",
    "Train MSE",
    "Validation MSE",
    "Train RMSE",
    "Validation RMSE",
    "Train R2",
    "Validation R2",
    "RMSE Gap"
]

print("RIDGE REGRESSION – ERROR COMPARISON")

display(
    ridge_results_df[regression_columns].round(4)
)

RIDGE REGRESSION – ERROR COMPARISON


,Experiment,Alpha,Train MAE,Validation MAE,Train MSE,Validation MSE,Train RMSE,Validation RMSE,Train R2,Validation R2,RMSE Gap
0,Ridge-4,10.00,1.0956,1.2920,2.4692,3.4597,1.5695,1.8322,0.8824,0.8328,0.2626
1,Ridge-3,1.00,1.1202,1.3597,2.4073,3.6059,1.5496,1.8718,0.8854,0.8259,0.3223
2,Ridge-2,0.10,1.1245,1.3710,2.4062,3.6438,1.5492,1.8817,0.8855,0.8241,0.3325
3,Ridge-1,0.01,1.1250,1.3722,2.4062,3.6481,1.5492,1.8828,0.8855,0.8239,0.3336
4,Ridge-5,100.00,1.3125,1.4287,3.9156,4.6342,1.9778,2.1210,0.8131,0.7765,0.1432


Grade prediction accuracy comparison table

In [19]:
grade_columns = [
    "Experiment",
    "Alpha",
    "Exact Grade Accuracy (%)",
    "Within 1 Grade (%)",
    "Within 2 Grades (%)"
]

print("RIDGE REGRESSION – GRADE ACCURACY")

display(
    ridge_results_df[grade_columns].round(2)
)

RIDGE REGRESSION – GRADE ACCURACY


,Experiment,Alpha,Exact Grade Accuracy (%),Within 1 Grade (%),Within 2 Grades (%)
0,Ridge-4,10.00,30.38,53.80,82.91
1,Ridge-3,1.00,30.38,51.27,79.43
2,Ridge-2,0.10,30.38,51.90,78.80
3,Ridge-1,0.01,30.38,51.90,78.80
4,Ridge-5,100.00,30.06,52.53,79.11


Classification metrics comparison table

In [20]:
classification_columns = [
    "Experiment",
    "Alpha",
    "Pass/Fail Accuracy (%)",
    "Precision",
    "Recall",
    "Specificity",
    "F1 Score"
]

print("RIDGE REGRESSION – PASS/FAIL CLASSIFICATION")

display(
    ridge_results_df[classification_columns].round(4)
)

RIDGE REGRESSION – PASS/FAIL CLASSIFICATION


,Experiment,Alpha,Pass/Fail Accuracy (%),Precision,Recall,Specificity,F1 Score
0,Ridge-4,10.00,85.4430,0.9771,0.8028,0.9612,0.8814
1,Ridge-3,1.00,86.0759,0.9774,0.8122,0.9612,0.8872
2,Ridge-2,0.10,86.0759,0.9774,0.8122,0.9612,0.8872
3,Ridge-1,0.01,86.0759,0.9774,0.8122,0.9612,0.8872
4,Ridge-5,100.00,86.7089,0.9942,0.8075,0.9903,0.8912


Confusion matrix comparison table

In [21]:
confusion_columns = [
    "Experiment",
    "Alpha",
    "TN",
    "FP",
    "FN",
    "TP"
]

print("RIDGE REGRESSION – CONFUSION MATRIX COMPARISON")

display(
    ridge_results_df[confusion_columns]
)

RIDGE REGRESSION – CONFUSION MATRIX COMPARISON


,Experiment,Alpha,TN,FP,FN,TP
0,Ridge-4,10.00,99,4,42,171
1,Ridge-3,1.00,99,4,40,173
2,Ridge-2,0.10,99,4,40,173
3,Ridge-1,0.01,99,4,40,173
4,Ridge-5,100.00,102,1,41,172


Check overfitting and underfitting

In [22]:
baseline_scores = cross_validate(
    DummyRegressor(strategy="mean"),
    X_train,
    y_train,
    cv=cv,
    scoring=scoring
)

baseline_rmse = np.sqrt(
    -baseline_scores["test_MSE"]
).mean()

def check_model_fitting(row):

    gap_ratio = (
        row["RMSE Gap"] / row["Train RMSE"]
        if row["Train RMSE"] > 0 else np.inf
    )

    if row["Validation RMSE"] >= baseline_rmse:
        return "Weak performance / possible underfitting"

    elif gap_ratio > 0.30:
        return "Possible overfitting"

    else:
        return "Reasonable generalization"

ridge_results_df["Fitting Assessment"] = (
    ridge_results_df.apply(check_model_fitting, axis=1)
)

print("Baseline RMSE:", round(baseline_rmse, 4))

display(
    ridge_results_df[
        [
            "Experiment",
            "Alpha",
            "Train RMSE",
            "Validation RMSE",
            "RMSE Gap",
            "Fitting Assessment"
        ]
    ].round(4)
)

Baseline RMSE: 4.5635


,Experiment,Alpha,Train RMSE,Validation RMSE,RMSE Gap,Fitting Assessment
0,Ridge-4,10.00,1.5695,1.8322,0.2626,Reasonable generalization
1,Ridge-3,1.00,1.5496,1.8718,0.3223,Reasonable generalization
2,Ridge-2,0.10,1.5492,1.8817,0.3325,Reasonable generalization
3,Ridge-1,0.01,1.5492,1.8828,0.3336,Reasonable generalization
4,Ridge-5,100.00,1.9778,2.1210,0.1432,Reasonable generalization


Final complete comparison table

In [23]:
final_comparison = ridge_results_df.copy()

final_comparison.insert(
    0,
    "Rank",
    np.arange(1, len(final_comparison) + 1)
)

print("=" * 65)
print("FINAL RIDGE REGRESSION COMPARISON")
print("=" * 65)

display(final_comparison.round(4))

FINAL RIDGE REGRESSION COMPARISON


,Rank,Experiment,Alpha,Train MAE,Validation MAE,Train MSE,Validation MSE,Train RMSE,Validation RMSE,Train R2,...,Pass/Fail Accuracy (%),Precision,Recall,Specificity,F1 Score,TP,TN,FP,FN,Fitting Assessment
0,1,Ridge-4,10.00,1.0956,1.2920,2.4692,3.4597,1.5695,1.8322,0.8824,...,85.4430,0.9771,0.8028,0.9612,0.8814,171,99,4,42,Reasonable generalization
1,2,Ridge-3,1.00,1.1202,1.3597,2.4073,3.6059,1.5496,1.8718,0.8854,...,86.0759,0.9774,0.8122,0.9612,0.8872,173,99,4,40,Reasonable generalization
2,3,Ridge-2,0.10,1.1245,1.3710,2.4062,3.6438,1.5492,1.8817,0.8855,...,86.0759,0.9774,0.8122,0.9612,0.8872,173,99,4,40,Reasonable generalization
3,4,Ridge-1,0.01,1.1250,1.3722,2.4062,3.6481,1.5492,1.8828,0.8855,...,86.0759,0.9774,0.8122,0.9612,0.8872,173,99,4,40,Reasonable generalization
4,5,Ridge-5,100.00,1.3125,1.4287,3.9156,4.6342,1.9778,2.1210,0.8131,...,86.7089,0.9942,0.8075,0.9903,0.8912,172,102,1,41,Reasonable generalization


Select the best Ridge Regression configuration

In [24]:
best_result = final_comparison.iloc[0]

best_name = best_result["Experiment"]
best_alpha = float(best_result["Alpha"])

print("=" * 60)
print("BEST RIDGE REGRESSION CONFIGURATION")
print("=" * 60)

print("Experiment:", best_name)
print("Alpha:", best_alpha)

print("\nREGRESSION METRICS")
print(f"Validation RMSE : {best_result['Validation RMSE']:.4f}")
print(f"Validation MAE  : {best_result['Validation MAE']:.4f}")
print(f"Validation R²   : {best_result['Validation R2']:.4f}")
print(f"RMSE Gap        : {best_result['RMSE Gap']:.4f}")

print("\nGRADE ACCURACY")
print(
    f"Exact Accuracy  : "
    f"{best_result['Exact Grade Accuracy (%)']:.2f}%"
)
print(
    f"Within ±1 Grade : "
    f"{best_result['Within 1 Grade (%)']:.2f}%"
)
print(
    f"Within ±2 Grades: "
    f"{best_result['Within 2 Grades (%)']:.2f}%"
)

print("\nPASS/FAIL CLASSIFICATION")
print(
    f"Accuracy : "
    f"{best_result['Pass/Fail Accuracy (%)']:.2f}%"
)
print(f"Precision: {best_result['Precision']:.4f}")
print(f"Recall   : {best_result['Recall']:.4f}")
print(f"F1 Score : {best_result['F1 Score']:.4f}")
print(f"Specificity: {best_result['Specificity']:.4f}")

print("\nCONFUSION MATRIX")
print("TN:", int(best_result["TN"]))
print("FP:", int(best_result["FP"]))
print("FN:", int(best_result["FN"]))
print("TP:", int(best_result["TP"]))

print("\nFITTING ASSESSMENT")
print(best_result["Fitting Assessment"])

BEST RIDGE REGRESSION CONFIGURATION
Experiment: Ridge-4
Alpha: 10.0

REGRESSION METRICS
Validation RMSE : 1.8322
Validation MAE  : 1.2920
Validation R²   : 0.8328
RMSE Gap        : 0.2626

GRADE ACCURACY
Exact Accuracy  : 30.38%
Within ±1 Grade : 53.80%
Within ±2 Grades: 82.91%

PASS/FAIL CLASSIFICATION
Accuracy : 85.44%
Precision: 0.9771
Recall   : 0.8028
F1 Score : 0.8814
Specificity: 0.9612

CONFUSION MATRIX
TN: 99
FP: 4
FN: 42
TP: 171

FITTING ASSESSMENT
Reasonable generalization
